# Seeing what the mountain sees

Avalanche control teams, backcountry guides, and ski patrol all face the same question: from this spot, what can I actually see?

A slope hidden behind a ridge is one you can't monitor or trigger remotely. Viewshed analysis works this out from elevation data. Given a point and an observer height, it checks which cells have an unblocked line of sight.

For more on avalanche terrain assessment, see [avalanche.org](https://avalanche.org).

### What you'll build

You'll calculate viewshed from observer points on two terrains, then map out what's visible and what isn't.

[Simple viewshed](#Simple-viewshed) · [Viewshed on terrain](#Viewshed-on-terrain)

We use numpy, pandas, and xarray for data handling, matplotlib for overlay plots, and xrspatial for the viewshed computation.

In [ ]:
import numpy as np
import pandas as pd
import xarray as xr

import matplotlib.pyplot as plt
from matplotlib.colors import ListedColormap

import xrspatial

## Simple viewshed

We start with a synthetic peak from a 2D normal distribution, place an observer off to one side, and check what's visible. The orange dot is the observer.

In [ ]:
W = 800
H = 600

OBSERVER_X = -12.5
OBSERVER_Y = 10

x_range = (-20, 20)
y_range = (-20, 20)

normal_df = pd.DataFrame({
   'x': np.random.normal(.5, 1, 10000000),
   'y': np.random.normal(.5, 1, 10000000)
})

counts, xedges, yedges = np.histogram2d(
    normal_df['x'].values, normal_df['y'].values,
    bins=[W, H], range=[list(x_range), list(y_range)]
)
normal_agg = xr.DataArray(
    counts.T.astype('float64'),
    dims=['y', 'x'],
    coords={
        'y': (yedges[:-1] + yedges[1:]) / 2,
        'x': (xedges[:-1] + xedges[1:]) / 2,
    }
)

normal_illuminated = normal_agg.xrs.hillshade()

fig, ax = plt.subplots(figsize=(10, 7.5))
normal_illuminated.plot.imshow(ax=ax, cmap='gray', alpha=128/255, add_colorbar=False)
ax.scatter([OBSERVER_X], [OBSERVER_Y], c='orange', s=100, zorder=5, edgecolors='white')
ax.set_axis_off()

### Calculate viewshed from the observer

Viewshed marks each cell visible or not. Visible cells show up in red.

In [ ]:
# Takes a moment
%time view = normal_agg.xrs.viewshed(x=OBSERVER_X, y=OBSERVER_Y)

visible = view.where(view >= 0)

fig, ax = plt.subplots(figsize=(10, 7.5))
normal_illuminated.plot.imshow(ax=ax, cmap='gray', alpha=128/255, add_colorbar=False)
visible.plot.imshow(ax=ax, cmap=ListedColormap(['red']), alpha=128/255, add_colorbar=False)
ax.scatter([OBSERVER_X], [OBSERVER_Y], c='orange', s=100, zorder=5, edgecolors='white')
ax.set_axis_off()

Red is visible, everything else is blocked. The peak throws a shadow behind it.

## Viewshed on terrain

Same idea on more realistic terrain. We generate a mountain range and place an observer at the center. Which slopes could they watch from that position?

In [ ]:
terrain = xr.DataArray(np.zeros((H, W)))
terrain = terrain.xrs.generate_terrain(
    x_range=(-250, 250), y_range=(-250, 250),
    zfactor=6000, warp_strength=0.4,
)

illuminated = terrain.xrs.hillshade()

OBSERVER_X = 0.0
OBSERVER_Y = 0.0

fig, ax = plt.subplots(figsize=(10, 7.5))
illuminated.plot.imshow(ax=ax, cmap='gray', alpha=128/255, add_colorbar=False)
terrain.plot.imshow(ax=ax, cmap='gray', alpha=128/255, add_colorbar=False)
ax.scatter([OBSERVER_X], [OBSERVER_Y], c='orange', s=100, zorder=5, edgecolors='white')
ax.set_axis_off()

### Observer elevation

`observer_elev` is the observer's height above ground in meters. We use 5 here, about standing height on a small platform. Lower values mean more blind spots behind ridgelines.

In [ ]:
view = terrain.xrs.viewshed(x=OBSERVER_X, y=OBSERVER_Y, observer_elev=5)

visible = view.where(view >= 0)

fig, ax = plt.subplots(figsize=(10, 7.5))
illuminated.plot.imshow(ax=ax, cmap='gray', alpha=128/255, add_colorbar=False)
terrain.plot.imshow(ax=ax, cmap='terrain', alpha=128/255, add_colorbar=False)
visible.plot.imshow(ax=ax, cmap=ListedColormap(['fuchsia']), alpha=0.5, add_colorbar=False)
ax.scatter([OBSERVER_X], [OBSERVER_Y], c='orange', s=100, zorder=5, edgecolors='white')
ax.set_axis_off()

Fuchsia is visible from the observer. The gaps behind ridges are blind spots.

### References

- [avalanche.org](https://avalanche.org): avalanche education, forecasts, and encyclopedia
- [Surface toolset overview](https://pro.arcgis.com/en/pro-app/tool-reference/spatial-analyst/an-overview-of-the-surface-tools.htm), Esri ArcGIS Pro